# 3주차 예제 — 따릉이 대여량 예측 (Decision Tree)

지난주와 **같은 데이터와 같은 X/y**를 사용합니다. 입력과 평가 구간을 유지한 채 모델만 선형회귀에서 Decision Tree로 바꾸면 두 모델의 차이를 같은 기준으로 비교할 수 있습니다.

1. Decision Tree란 무엇인가 (간단한 예제로 원리부터)
2. 지난주 데이터 재구성
3. 선형회귀 성능 다시 확인 (기준점)
4. Decision Tree — 제한 없이 학습 → 과적합 확인
5. 트리 시각화
6. 가지치기(pruning)로 과적합 줄이기
7. Feature Importance
8. 세 모델 종합 비교


## Part 1. Decision Tree란 무엇인가 (간단한 예제)

**Decision Tree(의사결정나무)**는 '예/아니오' 질문을 반복해 데이터를 점점 비슷한 그룹으로 나누는 방법입니다. 스무고개처럼 "기온이 22.5도보다 높습니까?"라는 질문으로 데이터를 두 그룹으로 나누고, 각 그룹을 다시 세분합니다. 더 나누지 않는 지점인 **잎(leaf)**에서는 그 잎에 속한 훈련 데이터의 평균을 예측값으로 사용합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import root_mean_squared_error

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
toy = pd.DataFrame({
    '기온': [5, 10, 15, 20, 25, 30],
    '판매량': [20, 35, 40, 55, 70, 90],  # 아이스크림 판매량
})

tree_toy = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_toy.fit(toy[['기온']], toy['판매량'])

print(export_text(tree_toy, feature_names=['기온']))


글로 읽으면 "기온이 22.5 이하입니까? → 그중에서 17.5 이하입니까? → 이 잎의 예측값은 31.67입니다"와 같이 조건이 중첩됩니다. 이어지는 그림에서 같은 경로를 시각적으로 확인할 수 있습니다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_tree(tree_toy, feature_names=['기온'], filled=True, ax=ax)
plt.show()


**잎(leaf)의 값이 해당 훈련 표본의 평균인지 직접 검산합니다.** `기온 <= 17.5`에 해당하는 데이터는 기온 5·10·15, 판매량 20·35·40입니다.


In [ ]:
print('기온<=17.5 그룹 평균:', toy[toy['기온'] <= 17.5]['판매량'].mean())
print('트리가 예측한 값 :', tree_toy.predict(pd.DataFrame({'기온': [10]}))[0])


두 값이 일치합니다. **회귀 트리의 잎 예측값은 그 잎에 도달한 훈련 타깃의 평균입니다.** 분기 지점은 각 후보로 나누었을 때 **두 그룹 내부의 제곱오차가 가장 많이 줄어드는 위치**를 기준으로 선택합니다. 이는 오차를 작게 만든다는 점에서 회귀 평가의 방향과 연결되지만, 매 분기에서 직접 계산하는 값은 해당 훈련 노드의 불순도 감소량입니다.


### 조건 없이 학습하면 어떻게 됩니까? — 과적합 미리보기

`max_depth`를 지정하지 않으면 트리는 각 관측치가 작은 잎에 들어갈 때까지 계속 세분될 수 있습니다. 이 경우 훈련 데이터의 세부 패턴에 지나치게 맞춰질 가능성이 있습니다.


In [ ]:
tree_unlimited = DecisionTreeRegressor(random_state=42).fit(toy[['기온']], toy['판매량'])
print('제한 없는 트리 예측값:', tree_unlimited.predict(toy[['기온']]))
print('실제 판매량        :', toy['판매량'].tolist())


훈련 예측값이 실제값과 완전히 같습니다. 그러나 훈련 데이터에 대한 높은 적합도가 새로운 기온값의 정확한 예측을 보장하지는 않습니다. 이처럼 훈련 데이터에는 매우 잘 맞지만 새 데이터에서 성능이 낮아질 수 있는 상태를 **과적합(overfitting)**이라고 합니다. `max_depth`, `min_samples_leaf` 등으로 트리의 복잡도를 조절하는 방법을 **사전 가지치기(pre-pruning)**라고 부릅니다. Part 4~6에서 실제 데이터의 훈련·평가 RMSE를 비교해 같은 현상을 확인합니다.


## Part 2. 지난주 데이터 재구성

2주차와 동일한 전처리를 사용합니다. 원본은 [서울 열린데이터광장 — 서울시 공공자전거 따릉이 이용현황(일별 대여건수)](https://data.seoul.go.kr/dataList/OA-14994/A/1/datasetView.do)에서 내려받을 수 있습니다. 이 노트북은 `dataset/extracted/따릉이 공공데이터/02_이용정보/`의 `서울특별시 공공자전거 일별 대여건수_*.csv` 파일을 읽습니다.


In [ ]:
import glob

base = '../dataset/extracted/따릉이 공공데이터/02_이용정보'
files = sorted(glob.glob(base + '/서울특별시 공공자전거 일별 대여건수_*.csv'))
dfs = [pd.read_csv(f, encoding='cp949') for f in files]
df = pd.concat(dfs, ignore_index=True)
df['대여일자'] = pd.to_datetime(df['대여일자'])
df = df.sort_values('대여일자').reset_index(drop=True)

df['day_index'] = (df['대여일자'] - df['대여일자'].min()).dt.days
df['월'] = df['대여일자'].dt.month
df['요일'] = df['대여일자'].dt.day_name()

month_dummies = pd.get_dummies(df['월'], prefix='월', drop_first=True)
weekday_dummies = pd.get_dummies(df['요일'], prefix='요일', drop_first=True)
X = pd.concat([df[['day_index']], month_dummies, weekday_dummies], axis=1)
y = df['대여건수']

split_idx = len(df) - 60
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
X_train.shape, X_test.shape


## Part 3. 선형회귀 성능 다시 확인 (기준점)

이번 주 비교의 기준선(baseline)으로 삼습니다.


In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression().fit(X_train, y_train)
rmse_linear = root_mean_squared_error(y_test, linear_model.predict(X_test))
print('선형회귀 RMSE:', rmse_linear)


## Part 4. Decision Tree — 제한 없이 학습

Part 1의 간단한 예제에서 확인한 상황을 실제 데이터에 적용합니다. 훈련 RMSE와 테스트 RMSE를 나란히 비교하면 과적합 가능성을 살펴볼 수 있습니다.


In [ ]:
tree_full = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

rmse_full_train = root_mean_squared_error(y_train, tree_full.predict(X_train))
rmse_full_test = root_mean_squared_error(y_test, tree_full.predict(X_test))
print('제한 없는 트리 - train RMSE:', rmse_full_train)
print('제한 없는 트리 - test RMSE :', rmse_full_test)


훈련 RMSE는 0에 가깝지만 테스트 RMSE는 선형회귀보다 높습니다. 이는 간단한 예제에서 확인한 것과 같은 패턴입니다. 훈련 데이터에는 매우 잘 맞지만 처음 보는 데이터에서는 오차가 더 커져, 일반화 성능을 함께 점검할 필요가 있음을 보여 줍니다.


## Part 5. 트리 시각화

방금 만든 트리는 깊어서 한 화면에서 구조를 읽기 어렵습니다. 따라서 **시각화 전용으로 깊이를 3단계로 제한한 별도 트리**를 만들어 분기 원리를 살펴봅니다. 이 트리는 설명용이며, 성능 비교에는 앞서 학습한 트리를 사용합니다.


In [ ]:
tree_for_viz = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(tree_for_viz, feature_names=X.columns, filled=True, ax=ax, fontsize=8)
plt.show()


맨 위 노드인 뿌리부터 어떤 변수가 분기에 사용되는지 살펴봅니다. 상위 분기에 `day_index`가 반복해서 등장한다면 이 트리가 시점 정보를 이용해 대여량 차이를 크게 줄였다는 뜻입니다. 이는 모델의 예측 방식에 대한 설명이며, 시간이 대여량의 원인이라는 의미는 아닙니다.


## Part 6. 가지치기(pruning)로 과적합 줄이기

Part 1에서 미리 본 것처럼, `max_depth`(최대 깊이)와 `min_samples_leaf`(잎 하나에 최소 몇 개 데이터가 남아야 하는지)로 트리 성장을 제한합니다.


In [ ]:
tree_pruned = DecisionTreeRegressor(max_depth=6, min_samples_leaf=20, random_state=42).fit(X_train, y_train)

rmse_pruned_train = root_mean_squared_error(y_train, tree_pruned.predict(X_train))
rmse_pruned_test = root_mean_squared_error(y_test, tree_pruned.predict(X_test))
print('가지치기 트리 - train RMSE:', rmse_pruned_train)
print('가지치기 트리 - test RMSE :', rmse_pruned_test)


train/test RMSE의 격차가 줄었고, 테스트 RMSE도 제한 없는 트리보다 낮아졌습니다. 가지치기는 훈련 적합을 일부 줄이는 대신 새 데이터에서의 오차와 일반화 격차를 낮출 수 있는 트레이드오프입니다.


## Part 7. Feature Importance

회귀에서는 계수(coefficient)로 다른 입력을 고정했을 때의 조건부 차이를 살펴봤습니다. 트리의 `feature_importances_`는 변수가 분기에서 만든 불순도 감소량을 합산한 값입니다. 계수와 달리 **부호(+/-)가 없으며 방향이나 인과관계를 알려 주는 값이 아니라는 점**을 함께 고려합니다.


In [ ]:
importances = pd.Series(tree_pruned.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
importances.head(10).plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Decision Tree Feature Importance (상위 10개)')
plt.show()


## Part 8. 세 모델 종합 비교

선형회귀 / 제한 없는 트리 / 가지치기 트리, 세 모델의 테스트 RMSE를 한 표로 비교합니다.


In [ ]:
comparison = pd.DataFrame({
    '모델': ['선형회귀', '트리(제한없음)', '트리(가지치기)'],
    'test RMSE': [rmse_linear, rmse_full_test, rmse_pruned_test],
})
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df['대여일자'].iloc[split_idx:], y_test.values, label='실제')
ax.plot(df['대여일자'].iloc[split_idx:], linear_model.predict(X_test), label='선형회귀')
ax.plot(df['대여일자'].iloc[split_idx:], tree_pruned.predict(X_test), label='가지치기 트리')
ax.legend()
ax.set_title('실제 vs 선형회귀 vs 가지치기 트리')
plt.xticks(rotation=45)
plt.show()


**왜 이번 구간에서 트리의 RMSE가 선형회귀보다 높았습니까?** 테스트 구간의 `day_index`는 훈련 범위보다 큽니다. 트리는 새로운 값이 들어오면 훈련 때 만든 잎 가운데 하나의 평균을 반환하므로 **추세를 범위 밖으로 연장하는 외삽 기능이 없습니다.** 반면 선형회귀는 학습한 직선을 범위 밖까지 연장할 수 있어, 이번처럼 추세가 있는 평가 구간에서 더 낮은 RMSE를 보였습니다.

이 사례는 모델의 복잡도만으로 성능을 결정할 수 없음을 보여 줍니다. 데이터의 구조와 평가 구간에 따라 더 단순한 모델이 더 낮은 오차를 보일 수 있으므로, 같은 분할과 지표로 비교한 뒤 선택합니다.


## 인사이트 정리 (예시)

- 제한 없는 트리는 훈련 RMSE가 0까지 낮아지지만 테스트 RMSE는 선형회귀보다 높았습니다. Part 1의 예제와 마찬가지로 과적합 가능성을 보여 주는 패턴입니다.
- 가지치기(`max_depth`, `min_samples_leaf`)를 적용하자 train/test RMSE 격차와 테스트 RMSE가 모두 줄었습니다.
- 트리의 최상위 분기와 feature importance에서 `day_index`가 크게 나타났습니다. 이는 트리가 시점 정보에 많이 의존했음을 뜻하며, 인과관계나 효과의 방향을 의미하지는 않습니다.
- 이번 평가 구간에서는 선형회귀의 테스트 RMSE가 더 낮았습니다. 트리는 훈련 범위 밖의 추세를 외삽하지 못하므로, **모델은 문제의 구조와 검증 결과를 함께 살펴 선택합니다.**

전력사용량 예측 과제에서도 '트리 학습 → 선형회귀와 성능 비교'를 진행합니다. 이번 과제는 결측치 처리 방법이 성능에 큰 영향을 줄 수 있는 데이터이므로, 처리 근거와 전후 결과를 함께 기록합니다.
